# UTSM: retrain the Random Forest with Laval data
**Upload this notebook to Colab, choose a CPU runtime, then Run all.**
When prompted, upload **Laval All Telem.zip**. No separate Python files or GitHub login are needed.

This notebook contains the reviewed training code, analytical reference and dependency list.
It prepares telemetry, tunes forests using September 19 sessions, tests on September 20 sessions,
compares against analytical models, and downloads the complete result bundle.

**Default results are exploratory:** current calibration is unverified and the archive records constant voltage.
The forest is not assumed to improve on the analytical model. Read the report before choosing a model.


## 1. Install the pinned CPU environment
Run this first in a fresh runtime. If Colab reports that an imported package changed, restart the runtime and Run all again.
No GPU is required for the forest or analytical model.


In [ ]:
import os, sys, subprocess
from pathlib import Path
REQUIREMENTS = 'numpy==1.26.4\npandas==2.2.3\nscikit-learn==1.6.1\nmatplotlib==3.9.4\njoblib==1.4.2\n'
IN_COLAB = 'google.colab' in sys.modules or Path('/content').is_dir()
WORK_DIR = Path(os.environ.get('UTSM_WORK_DIR', '/content/utsm-laval' if IN_COLAB else 'laval-runtime')).resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / 'requirements-colab.txt').write_text(REQUIREMENTS, encoding='utf-8')
if os.environ.get('UTSM_SKIP_INSTALL') != '1':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORK_DIR / 'requirements-colab.txt')])
print('Workspace:', WORK_DIR)


## 2. Load the bundled, versioned implementation
This cell writes only the helper and frozen analytical reference into the runtime workspace.
It does not download source code. The SHA-256 check catches an accidental mismatch in the embedded helper.


In [ ]:
import hashlib, importlib.util, json
BUNDLED_TRAINING_SOURCE = '"""Reproducible, session-held-out Laval power-model training.\n\nPython 3.9+; see requirements-colab.txt for the Colab environment.\nRaw archives are read in memory, never extracted or modified.\nThis is an offline power/energy estimator, not a throttle controller.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport io\nimport json\nimport platform\nimport re\nimport subprocess\nimport sys\nimport zipfile\nfrom dataclasses import asdict, dataclass, field\nfrom pathlib import Path\nfrom typing import Any\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport sklearn\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import Ridge\nfrom sklearn.model_selection import GroupKFold\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nSCHEMA_VERSION = "laval-power-v1"\nCORE_FEATURES = [\n    "speed_kph", "speed_squared", "positive_accel", "negative_accel",\n    "positive_accel_speed", "negative_accel_speed",\n]\nENHANCED_FEATURES = CORE_FEATURES + [\n    "temperature_C", "temperature_missing", "speed_mean_3", "speed_std_3",\n    "wheel_speed_used",\n]\nREQUIRED = {"timestamp_ms", "current_mA", "voltage_mV"}\nCANDIDATES = [\n    {"name": "rf_core_leaf20", "features": "core", "min_samples_leaf": 20, "max_depth": 12},\n    {"name": "rf_enhanced_leaf20", "features": "enhanced", "min_samples_leaf": 20, "max_depth": 12},\n    {"name": "rf_enhanced_leaf5", "features": "enhanced", "min_samples_leaf": 5, "max_depth": 12},\n    {"name": "rf_enhanced_leaf50", "features": "enhanced", "min_samples_leaf": 50, "max_depth": None},\n]\n\n\n@dataclass\nclass Config:\n    # These defaults are explicitly exploratory, not a sensor calibration claim.\n    current_mode: str = "magnitude"  # magnitude | positive | negative\n    current_scale: float = 1.0\n    current_zero_mA: float = 0.0\n    calibration_confirmed: bool = False\n    calibration_overrides: dict[str, dict[str, Any]] = field(default_factory=dict)\n    test_dates: list[str] = field(default_factory=lambda: ["2026-09-20"])\n    max_gap_s: float = 2.0\n    min_speed_kph: float = 2.0\n    max_speed_kph: float = 60.0\n    max_abs_accel_m_s2: float = 4.0\n    max_abs_current_mA: float = 60000.0\n    max_hdop: float = 3.0\n    max_speed_disagreement_kph: float = 10.0\n    min_session_rows: int = 50\n    min_session_duration_s: float = 20.0\n    window_seconds: float = 10.0\n    min_window_duration_s: float = 5.0\n    n_estimators: int = 200\n    n_jobs: int = 2\n    seed: int = 42\n    refit_all_for_release: bool = False\n\n    def validate(self):\n        if self.current_mode not in {"magnitude", "positive", "negative"}:\n            raise ValueError("current_mode must be magnitude, positive or negative")\n        if not np.isfinite(self.current_scale) or self.current_scale <= 0:\n            raise ValueError("current_scale must be positive and finite")\n        if not np.isfinite(self.current_zero_mA):\n            raise ValueError("current_zero_mA must be finite")\n        if not self.test_dates or any(not re.fullmatch(r"\\d{4}-\\d{2}-\\d{2}", d) for d in self.test_dates):\n            raise ValueError("Provide explicit ISO test_dates before training")\n        positive = [self.max_gap_s, self.max_speed_kph, self.max_abs_accel_m_s2,\n                    self.max_abs_current_mA, self.max_hdop, self.max_speed_disagreement_kph,\n                    self.min_session_duration_s, self.window_seconds, self.min_window_duration_s]\n        if not all(np.isfinite(v) and v > 0 for v in positive):\n            raise ValueError("Time, speed and quality limits must be positive and finite")\n        if not 0 <= self.min_speed_kph < self.max_speed_kph:\n            raise ValueError("Invalid speed range")\n        if self.min_session_rows < 2 or self.n_estimators < 1 or self.n_jobs < 1:\n            raise ValueError("Invalid sample or forest configuration")\n        if self.min_window_duration_s > self.window_seconds:\n            raise ValueError("min_window_duration_s cannot exceed window_seconds")\n        if self.refit_all_for_release and not self.calibration_confirmed:\n            raise ValueError("Confirm calibration before requesting a release refit")\n\n\ndef sha256(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef number(df: pd.DataFrame, name: str, default=np.nan) -> pd.Series:\n    if name not in df:\n        return pd.Series(default, index=df.index, dtype=float)\n    return pd.to_numeric(df[name], errors="coerce")\n\n\ndef calibrated_current(values: pd.Series, cfg: Config, filename: str) -> pd.Series:\n    override = cfg.calibration_overrides.get(filename, {})\n    unknown = set(override) - {"current_mode", "current_scale", "current_zero_mA"}\n    if unknown:\n        raise ValueError(f"Unknown calibration keys for {filename}: {sorted(unknown)}")\n    mode = override.get("current_mode", cfg.current_mode)\n    scale = float(override.get("current_scale", cfg.current_scale))\n    zero = float(override.get("current_zero_mA", cfg.current_zero_mA))\n    if mode not in {"magnitude", "positive", "negative"} or not np.isfinite(scale) or scale <= 0 or not np.isfinite(zero):\n        raise ValueError(f"Invalid calibration for {filename}")\n    corrected = (values - zero) * scale\n    if mode == "magnitude":\n        return corrected.abs()\n    # Consumption only: regen is not silently counted as positive consumption.\n    return (corrected if mode == "positive" else -corrected).clip(lower=0)\n\n\ndef causal_features(df: pd.DataFrame) -> pd.DataFrame:\n    """Rows must be in source order, with blocks reset after outages/reboots."""\n    out = df.copy()\n    groups = out.groupby(["session_id", "block_id"], sort=False)\n    smooth = groups["raw_speed_kph"].transform(\n        lambda s: s.rolling(3, min_periods=1).median()\n    )\n    out["speed_kph"] = smooth\n    prev_speed = smooth.groupby([out.session_id, out.block_id], sort=False).shift()\n    out["accel_m_s2"] = (smooth - prev_speed) / (3.6 * out["dt_s"])\n    speed_groups = smooth.groupby([out.session_id, out.block_id], sort=False)\n    out["speed_mean_3"] = speed_groups.transform(lambda s: s.rolling(3, min_periods=1).mean())\n    out["speed_std_3"] = speed_groups.transform(lambda s: s.rolling(3, min_periods=1).std(ddof=0))\n    out["speed_squared"] = smooth ** 2\n    out["positive_accel"] = out.accel_m_s2.clip(lower=0)\n    out["negative_accel"] = (-out.accel_m_s2).clip(lower=0)\n    out["positive_accel_speed"] = out.positive_accel * smooth\n    out["negative_accel_speed"] = out.negative_accel * smooth\n    out["temperature_missing"] = out.temperature_C.isna().astype(float)\n    return out\n\n\ndef prepare_session(raw: pd.DataFrame, filename: str, cfg: Config) -> tuple[pd.DataFrame, dict]:\n    """Build valid observed-driving intervals without bridging invalid rows."""\n    missing = REQUIRED - set(raw.columns)\n    if missing:\n        raise ValueError(f"{filename}: missing columns {sorted(missing)}")\n    df = raw.copy().reset_index(drop=True)\n    stamp = re.match(r"(\\d{4}-\\d{2}-\\d{2})", Path(filename).name)\n    if not stamp:\n        raise ValueError(f"{filename}: date must begin YYYY-MM-DD")\n    session = Path(filename).stem\n    df["session_id"] = session\n    df["date"] = stamp.group(1)\n    df["source_row"] = np.arange(len(df))\n    ts = number(df, "timestamp_ms")\n    current_raw = number(df, "current_mA")\n    current = calibrated_current(current_raw, cfg, Path(filename).name)\n    voltage = number(df, "voltage_mV") / 1000.0\n    dt = ts.diff() / 1000.0\n    temporal = dt.gt(0) & dt.le(cfg.max_gap_s) & ts.notna() & ts.shift().notna()\n\n    wheel = number(df, "wheel_speed_kmph")\n    if "wheel_speed_kph" in df:\n        wheel = number(df, "wheel_speed_kph")\n    gps = number(df, "gps_speed_kmph")\n    wheel_ok = number(df, "wheel_speed_valid", 0).eq(1) & wheel.between(0, cfg.max_speed_kph)\n    lat = number(df, "gps_lat")\n    lon = number(df, "gps_long")\n    gps_ok = (number(df, "gps_location_valid", 0).eq(1)\n              & gps.between(0, cfg.max_speed_kph)\n              & lat.between(-90, 90) & lon.between(-180, 180)\n              & (lat.abs() + lon.abs()).gt(0)\n              & number(df, "gps_hdop").between(0, cfg.max_hdop))\n    if "gps_time_valid" in df:\n        gps_ok &= number(df, "gps_time_valid", 0).eq(1)\n    if "gps_time_utc" in df:\n        clock = df["gps_time_utc"].astype(str)\n        if "gps_centisecond" in df:\n            clock = clock + ":" + df["gps_centisecond"].astype(str)\n        clock_changed = clock.ne(clock.shift()) | dt.le(0)\n        last_clock_update_ms = ts.where(clock_changed).ffill()\n        gps_ok &= (ts - last_clock_update_ms).between(0, cfg.max_gap_s * 1000)\n    # Reject disagreement instead of choosing the sensor that makes a model win.\n    disagreement = wheel_ok & gps_ok & (wheel - gps).abs().gt(cfg.max_speed_disagreement_kph)\n    speed = wheel.where(wheel_ok, gps.where(gps_ok)).mask(disagreement)\n    temperature = number(df, "temperature_C")\n    if "motor_temperature_C" in df:\n        temperature = number(df, "motor_temperature_C")\n    temp_flag = "motor_temperature_valid" if "motor_temperature_valid" in df else "temperature_valid"\n    temperature = temperature.where(number(df, temp_flag, 0).eq(1) & temperature.between(-40, 180))\n    electrical = (current_raw.abs().le(cfg.max_abs_current_mA)\n                  & voltage.between(18, 30) & current.notna())\n    sensor_ok = electrical & speed.notna()\n    # Every invalid reading or timestamp discontinuity begins a new causal block.\n    boundaries = (~temporal) | (~sensor_ok) | (~sensor_ok.shift(fill_value=False))\n    block = boundaries.cumsum()\n    df["block_id"] = block\n    df["time_s"] = ts / 1000.0\n    df["dt_s"] = dt\n    df["raw_speed_kph"] = speed\n    df["temperature_C"] = temperature\n    df["wheel_speed_used"] = wheel_ok.astype(float)\n    df["corrected_current_mA"] = current\n    df["voltage_v"] = voltage\n    power = current / 1000.0 * voltage\n    # Target is interval-average power; integration uses the same intervals for all models.\n    df["target_power_w"] = (power + power.shift()) / 2.0\n    valid_interval = temporal & sensor_ok & sensor_ok.shift(fill_value=False)\n    df = causal_features(df)\n    valid = (valid_interval & df.speed_kph.between(cfg.min_speed_kph, cfg.max_speed_kph)\n             & df.accel_m_s2.abs().le(cfg.max_abs_accel_m_s2))\n    df["actual_energy_j"] = df.target_power_w * df.dt_s\n    finite = np.isfinite(df[CORE_FEATURES + ["target_power_w", "dt_s"]].to_numpy()).all(axis=1)\n    valid &= finite\n    kept = df.loc[valid].copy()\n    # Separate report groups at every excluded interval; no invented whole laps.\n    continuity = kept.source_row.diff().ne(1) | kept.block_id.diff().ne(0)\n    kept["evaluation_block"] = continuity.cumsum().astype(int)\n    report = {\n        "filename": filename, "session_id": session, "date": stamp.group(1),\n        "raw_rows": len(raw), "accepted_rows": len(kept),\n        "accepted_duration_s": float(kept.dt_s.sum()),\n        "file_span_s": float((ts.max() - ts.min()) / 1000) if ts.notna().any() else 0,\n        "negative_current_rows": int(current_raw.lt(0).sum()),\n        "gps_valid_rows": int(gps_ok.sum()), "wheel_valid_rows": int(wheel_ok.sum()),\n        "speed_disagreement_rows": int(disagreement.sum()),\n        "timestamp_reset_or_duplicate_rows": int(dt.le(0).sum()),\n        "gap_rows": int(dt.gt(cfg.max_gap_s).sum()),\n        "constant_voltage": bool(voltage.nunique(dropna=True) <= 1),\n        "voltage_median_v": float(voltage.median()) if voltage.notna().any() else None,\n        "status": "accepted",\n    }\n    if len(kept) < cfg.min_session_rows or kept.dt_s.sum() < cfg.min_session_duration_s:\n        report["status"] = "excluded_insufficient_valid_driving"\n        report["eligible_rows_before_session_filter"] = len(kept)\n        report["accepted_rows"] = 0\n        report["accepted_duration_s"] = 0.0\n        kept = kept.iloc[:0]\n    return kept, report\n\n\ndef load_archive(archive: str | Path, cfg: Config) -> tuple[pd.DataFrame, pd.DataFrame, dict]:\n    cfg.validate()\n    archive = Path(archive)\n    if archive.stat().st_size > 500_000_000:\n        raise ValueError("Archive exceeds 500 MB; split the archive before training")\n    source = archive.read_bytes()\n    if len(source) > 500_000_000:\n        raise ValueError("Archive exceeds 500 MB; split the archive before training")\n    frames, reports, hashes = [], [], {}\n    seen, seen_rows, session_ids = set(), set(), set()\n    with zipfile.ZipFile(io.BytesIO(source)) as z:\n        infos = [i for i in z.infolist() if i.filename.lower().endswith(".csv") and not i.is_dir()\n                 and not any(p.startswith(".") or p == "__MACOSX" for p in Path(i.filename).parts)]\n        if sum(i.file_size for i in infos) > 1_000_000_000:\n            raise ValueError("Uncompressed CSV contents exceed 1 GB")\n        for info in sorted(infos, key=lambda i: i.filename):\n            data = z.read(info)  # no path extraction\n            normalized = data.decode("utf-8-sig").replace("\\r\\n", "\\n").replace("\\r", "\\n")\n            digest = sha256(normalized.encode())\n            hashes[info.filename] = {"sha256": sha256(data), "normalized_sha256": digest, "bytes": len(data)}\n            if digest in seen:\n                reports.append({"filename": info.filename, "status": "excluded_duplicate_content"})\n                continue\n            seen.add(digest)\n            try:\n                raw = pd.read_csv(io.StringIO(normalized))\n            except (pd.errors.EmptyDataError, pd.errors.ParserError) as exc:\n                reports.append({"filename": info.filename, "status": "excluded_unreadable", "reason": str(exc)})\n                continue\n            semantic = sha256(json.dumps(raw.columns.tolist()).encode() + pd.util.hash_pandas_object(raw, index=False).to_numpy().tobytes())\n            hashes[info.filename]["parsed_rows_sha256"] = semantic\n            if semantic in seen_rows:\n                reports.append({"filename": info.filename, "status": "excluded_duplicate_content"})\n                continue\n            seen_rows.add(semantic)\n            if not REQUIRED.issubset(raw.columns):\n                reports.append({"filename": info.filename, "status": "excluded_missing_columns",\n                                "reason": str(sorted(REQUIRED - set(raw.columns)))})\n                continue\n            frame, report = prepare_session(raw, info.filename, cfg)\n            if report["session_id"] in session_ids:\n                raise ValueError("Duplicate session basename in archive: " + report["session_id"])\n            session_ids.add(report["session_id"])\n            reports.append(report)\n            if not frame.empty:\n                frames.append(frame)\n    unknown_overrides = set(cfg.calibration_overrides) - {Path(n).name for n in hashes}\n    if unknown_overrides:\n        raise ValueError("Calibration overrides name missing CSVs: " + str(sorted(unknown_overrides)))\n    if not frames:\n        raise ValueError("No usable driving intervals. Inspect calibration, sensor flags and data schema.")\n    frame = pd.concat(frames, ignore_index=True)\n    if frame[["session_id", "source_row"]].duplicated().any():\n        raise AssertionError("Duplicate source rows")\n    # A reboot must not create a separate training/test group for the same CSV.\n    provenance = {"archive": archive.name, "archive_sha256": sha256(source), "files": hashes}\n    return frame, pd.DataFrame(reports), provenance\n\n\ndef split_sessions(frame: pd.DataFrame, cfg: Config) -> tuple[pd.DataFrame, pd.DataFrame, dict]:\n    test = frame[frame.date.isin(cfg.test_dates)].copy()\n    dev = frame[~frame.date.isin(cfg.test_dates)].copy()\n    if test.empty:\n        raise ValueError("No valid test sessions on configured test_dates")\n    if dev.session_id.nunique() < 2:\n        raise ValueError("Need at least two usable development sessions for grouped tuning")\n    if dev.date.max() >= test.date.min():\n        raise ValueError("Development dates must precede test dates; do not randomly split rows")\n    if set(dev.session_id) & set(test.session_id):\n        raise AssertionError("Session leakage")\n    # Exact numeric rows can recur naturally at rest; duplicate timestamps/exports need stronger checks.\n    # Identical sequences across files are excluded above by normalized content hash.\n    manifest = {"development_sessions": sorted(dev.session_id.unique().tolist()),\n                "test_sessions": sorted(test.session_id.unique().tolist()),\n                "test_dates": cfg.test_dates, "development_rows": len(dev), "test_rows": len(test)}\n    return dev, test, manifest\n\n\ndef sample_weights(frame: pd.DataFrame) -> np.ndarray:\n    # Equal total weight per session, distributed by actual observed duration.\n    session_seconds = frame.groupby("session_id").dt_s.transform("sum").to_numpy()\n    weights = frame.dt_s.to_numpy() / session_seconds\n    return weights * len(frame) / frame.session_id.nunique()\n\n\ndef feature_names(kind: str) -> list[str]:\n    if kind == "core":\n        return list(CORE_FEATURES)\n    if kind == "enhanced":\n        return list(ENHANCED_FEATURES)\n    raise ValueError(f"Unknown feature set: {kind}")\n\n\ndef train_forest(frame: pd.DataFrame, candidate: dict, cfg: Config) -> dict:\n    names = feature_names(candidate["features"])\n    estimator = make_pipeline(\n        SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True),\n        RandomForestRegressor(n_estimators=cfg.n_estimators, random_state=cfg.seed,\n                              n_jobs=cfg.n_jobs, min_samples_leaf=candidate["min_samples_leaf"],\n                              max_depth=candidate["max_depth"]),\n    )\n    estimator.fit(frame[names], frame.target_power_w,\n                  randomforestregressor__sample_weight=sample_weights(frame))\n    return {"schema_version": SCHEMA_VERSION, "kind": "random_forest", "features": names,\n            "estimator": estimator, "candidate": candidate}\n\n\ndef train_analytical(frame: pd.DataFrame) -> dict:\n    # A matched-data analytical refit; distinct from the frozen Front Campus deployment artifact.\n    # Standardization parameters are learned only from this fold.\n    estimator = make_pipeline(StandardScaler(), Ridge(alpha=1.0))\n    estimator.fit(frame[CORE_FEATURES], frame.target_power_w,\n                  ridge__sample_weight=sample_weights(frame))\n    return {"schema_version": SCHEMA_VERSION, "kind": "analytical_refit",\n            "features": list(CORE_FEATURES), "estimator": estimator}\n\n\ndef predict_power(bundle: dict, feature_frame: pd.DataFrame) -> np.ndarray:\n    """Predict interval power from canonical causal features; no label columns required."""\n    if bundle.get("schema_version") != SCHEMA_VERSION:\n        raise ValueError("Unsupported feature schema")\n    missing = set(bundle["features"]) - set(feature_frame.columns)\n    if missing:\n        raise ValueError("Missing prediction features: " + str(sorted(missing)))\n    return np.maximum(bundle["estimator"].predict(feature_frame[bundle["features"]]), 0.0)\n\n\ndef predict_front_campus(frame: pd.DataFrame, reference: dict) -> np.ndarray:\n    """Frozen historical model electrical rule, no motor physics floor, neutral position.\n\n    Historical acceleration-action threshold is 1.0 in classify_strategy_action;\n    preserve it for the frozen reference rather than silently changing the incumbent.\n    Grade is zero: unreliable GPS altitude is not treated as surveyed grade.\n    """\n    speed = frame.speed_kph.to_numpy()\n    accel = frame.accel_m_s2.to_numpy()\n    pos, neg = np.maximum(accel, 0), np.maximum(-accel, 0)\n    action_acc = accel > 1.0\n    action_coast = accel < -1.0\n    action_hold = ~(action_acc | action_coast)\n    zeros = np.zeros(len(frame))\n    design = np.column_stack([np.ones(len(frame)), speed, speed**2, pos, neg, zeros, zeros,\n                              zeros, action_acc, action_hold, action_coast, pos*speed, neg*speed])\n    raw_current = np.maximum(design @ np.asarray(reference["current_coeffs"]), 0)\n    raw_power = np.maximum(design @ np.asarray(reference["power_coeffs"]), 0)\n    voltage = float(reference["median_voltage_v"])\n    on = np.full(len(frame), float(reference["on_current_mA"]))\n    cruise = float(reference["cruise_current_mA"])\n    duty_acc = np.clip(0.22 + pos*1.8, 0.18, 1.0)\n    on = np.where(action_acc, np.maximum(on, raw_current / np.maximum(duty_acc, 0.05)), on)\n    avg = np.where(action_acc, np.maximum(raw_current, on*duty_acc), np.maximum(raw_current, cruise))\n    on = np.where(action_hold, np.maximum(on, avg*1.35), on)\n    duty_hold = np.clip(avg/np.maximum(on, 1), 0.04, 0.85)\n    avg = np.where(action_hold, on*duty_hold, avg)\n    power = np.maximum(raw_power, avg/1000*voltage)\n    return np.where(action_coast, 0.0, power)\n\n\ndef evaluation(frame: pd.DataFrame, predicted_power: np.ndarray, name: str, cfg: Config):\n    if len(frame) != len(predicted_power):\n        raise ValueError("Prediction count does not match evaluation rows")\n    rows = frame[["session_id", "date", "source_row", "time_s", "block_id",\n                  "evaluation_block", "dt_s", "actual_energy_j", "target_power_w"]].copy()\n    rows["model"] = name\n    rows["predicted_power_w"] = np.asarray(predicted_power)\n    rows["predicted_energy_j"] = rows.predicted_power_w * rows.dt_s\n    rows["absolute_power_error_w"] = (rows.predicted_power_w - rows.target_power_w).abs()\n    grouped = rows.groupby(["model", "session_id", "date"], sort=True)\n    sessions = grouped.agg(actual_energy_j=("actual_energy_j", "sum"),\n                           predicted_energy_j=("predicted_energy_j", "sum"),\n                           valid_duration_s=("dt_s", "sum"),\n                           power_mae_w=("absolute_power_error_w", "mean")).reset_index()\n    denom = sessions.actual_energy_j.replace(0, np.nan)\n    sessions["signed_error_pct"] = (sessions.predicted_energy_j - sessions.actual_energy_j) / denom * 100\n    sessions["absolute_error_pct"] = sessions.signed_error_pct.abs()\n    rows["window"] = np.floor(rows.time_s / cfg.window_seconds).astype(int)\n    windows = rows.groupby(["model", "session_id", "evaluation_block", "window"], sort=True).agg(\n        actual_energy_j=("actual_energy_j", "sum"), predicted_energy_j=("predicted_energy_j", "sum"),\n        valid_duration_s=("dt_s", "sum")).reset_index()\n    windows = windows[windows.valid_duration_s >= cfg.min_window_duration_s].copy()\n    windows["absolute_error_pct"] = ((windows.predicted_energy_j - windows.actual_energy_j).abs()\n                                    / windows.actual_energy_j.replace(0, np.nan) * 100)\n    return sessions, windows, rows\n\n\ndef tune_forest(dev: pd.DataFrame, cfg: Config, candidates: list[dict] | None = None):\n    candidates = candidates or CANDIDATES\n    cv = GroupKFold(n_splits=min(3, dev.session_id.nunique()))\n    folds = list(cv.split(dev, groups=dev.session_id))\n    results = []\n    for candidate in candidates:\n        all_sessions = []\n        for fold, (train_index, val_index) in enumerate(folds):\n            train, val = dev.iloc[train_index], dev.iloc[val_index]\n            assert not set(train.session_id) & set(val.session_id)\n            bundle = train_forest(train, candidate, cfg)\n            metrics, _, _ = evaluation(val, predict_power(bundle, val), candidate["name"], cfg)\n            metrics["fold"] = fold\n            all_sessions.append(metrics)\n        metrics = pd.concat(all_sessions, ignore_index=True)\n        score = float(metrics.absolute_error_pct.mean())\n        if not np.isfinite(score):\n            raise ValueError("No finite development CV score; inspect zero-energy sessions")\n        results.append({"candidate": candidate["name"], "cv_mean_session_error_pct": score,\n                        "features": candidate["features"], "min_samples_leaf": candidate["min_samples_leaf"],\n                        "max_depth": candidate["max_depth"]})\n    result = pd.DataFrame(results).sort_values(["cv_mean_session_error_pct", "candidate"]).reset_index(drop=True)\n    selected = next(c for c in candidates if c["name"] == result.iloc[0].candidate)\n    return selected, result\n\n\ndef save_report(out: Path, summary: dict, sessions: pd.DataFrame):\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    pivot = sessions.pivot(index="session_id", columns="model", values="absolute_error_pct")\n    ax = pivot.plot.bar(figsize=(11, 5), ylabel="Absolute covered-driving energy error (%)",\n                        title="Held-out Laval sessions (lower is better)")\n    ax.set_xlabel("Session")\n    plt.xticks(rotation=25, ha="right")\n    plt.tight_layout()\n    plt.savefig(out / "comparison.png", dpi=160)\n    plt.close()\n    ranking = pd.DataFrame(summary["model_ranking"])\n    html = ("<!doctype html><html><head><meta charset=\'utf-8\'><title>Laval model report</title>"\n            "<style>body{max-width:1000px;margin:40px auto;font:16px Arial;color:#123}"\n            "td,th{padding:9px;border-bottom:1px solid #ccd}img{max-width:100%}</style></head><body>"\n            "<h1>Laval Random Forest vs analytical</h1><p><b>" + summary["status"] + "</b></p>"\n            "<p>Same held-out covered-driving intervals. These are not whole-lap errors or strategy savings.</p>"\n            + "<p><b>" + summary["recommendation"] + "</b></p>"\n            + ranking.to_html(index=False, float_format=lambda x: f"{x:.2f}")\n            + "<h2>Warnings</h2><ul>" + "".join("<li>" + w + "</li>" for w in summary["warnings"])\n            + "</ul><img src=\'comparison.png\'><h2>Per-session results</h2>"\n            + sessions.to_html(index=False, float_format=lambda x: f"{x:.2f}") + "</body></html>")\n    (out / "report.html").write_text(html, encoding="utf-8")\n\n\ndef run_workflow(archive: str | Path, output: str | Path, cfg: Config,\n                 reference_path: str | Path | None = None, candidates=None) -> dict:\n    cfg.validate()\n    out = Path(output)\n    if out.exists() and any(out.iterdir()):\n        raise FileExistsError("Choose an empty/new output directory; existing results are never overwritten")\n    out.mkdir(parents=True, exist_ok=True)\n    (out / "config.json").write_text(json.dumps(asdict(cfg), indent=2), encoding="utf-8")\n    frame, quality, provenance = load_archive(archive, cfg)\n    provenance["implementation_sha256"] = sha256(Path(__file__).read_bytes())\n    quality.to_csv(out / "data_quality.csv", index=False)\n    (out / "source_manifest.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")\n    frame.to_csv(out / "canonical_intervals.csv.gz", index=False, compression="gzip")\n    dev, test, split = split_sessions(frame, cfg)\n    (out / "split_manifest.json").write_text(json.dumps(split, indent=2), encoding="utf-8")\n    selected, cv_scores = tune_forest(dev, cfg, candidates)\n    cv_scores.to_csv(out / "development_cv.csv", index=False)\n    forest = train_forest(dev, selected, cfg)\n    analytical = train_analytical(dev)\n    # Do not inspect test scores until feature/model selection has completed.\n    predictions = {"random_forest": predict_power(forest, test),\n                   "analytical_refit": predict_power(analytical, test)}\n    if reference_path is not None:\n        reference_bytes = Path(reference_path).read_bytes()\n        reference = json.loads(reference_bytes)\n        predictions["frozen_front_campus"] = predict_front_campus(test, reference)\n        (out / "front_campus_reference.json").write_bytes(reference_bytes)\n        provenance["front_campus_reference_sha256"] = sha256(reference_bytes)\n        (out / "source_manifest.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")\n    session_tables, window_tables, prediction_tables = [], [], []\n    ranking = []\n    for name, pred in predictions.items():\n        sessions, windows, rows = evaluation(test, pred, name, cfg)\n        session_tables.append(sessions)\n        window_tables.append(windows)\n        prediction_tables.append(rows)\n        finite = sessions.absolute_error_pct.dropna()\n        if finite.empty:\n            raise ValueError("No positive-energy held-out sessions; percentage errors are undefined")\n        ranking.append({"model": name, "mean_session_error_pct": float(finite.mean()),\n                        "median_session_error_pct": float(finite.median()),\n                        "p90_session_error_pct": float(finite.quantile(0.9)),\n                        "mean_signed_session_error_pct": float(sessions.signed_error_pct.mean()),\n                        "window_energy_mae_wh": float((windows.predicted_energy_j-windows.actual_energy_j).abs().mean()/3600)\n                        if len(windows) else None})\n    sessions = pd.concat(session_tables, ignore_index=True)\n    sessions.to_csv(out / "held_out_sessions.csv", index=False)\n    pd.concat(window_tables, ignore_index=True).to_csv(out / "held_out_windows.csv", index=False)\n    pd.concat(prediction_tables, ignore_index=True).to_csv(out / "held_out_predictions.csv.gz", index=False, compression="gzip")\n    warnings = [\n        "Only September 19-20 Laval data; no untouched future driving day is established.",\n        "Errors cover accepted moving intervals, not whole laps or all recorded session energy.",\n        "No measured throttle channel: this estimator does not establish counterfactual strategy savings.",\n        "The analytical refit uses matched core features; it is not the historical deployment simulator.",\n        "Frozen Front Campus uses zero grade and no additional motor-physics floor; this is observed-driving replay.",\n        "This archive has already been explored in an audit; this is retrospective session-held-out evaluation, not a new prospective field test.",\n        "Repeated tuning after viewing these test errors invalidates the holdout."\n    ]\n    if not cfg.calibration_confirmed:\n        warnings.insert(0, "UNVERIFIED current calibration: all models and errors are exploratory only.")\n    if quality.get("constant_voltage", pd.Series(dtype=bool)).fillna(False).any():\n        warnings.insert(0, "Constant recorded voltage detected: energy labels assume recorded voltage is correct; battery sag is unobserved.")\n    status = "CALIBRATION CONFIRMED BY USER; FIELD VALIDATION STILL REQUIRED" if cfg.calibration_confirmed else "EXPLORATORY - CALIBRATION NOT CONFIRMED"\n    forest.update({"config": asdict(cfg), "source_manifest": provenance, "split_manifest": split,\n                   "status": status, "training_scope": "development_sessions_only"})\n    analytical.update({"config": asdict(cfg), "status": status, "training_scope": "development_sessions_only"})\n    joblib.dump(forest, out / "random_forest_development.joblib")\n    joblib.dump(analytical, out / "analytical_development.joblib")\n    (out / "feature_schema.json").write_text(json.dumps({\n        "version": SCHEMA_VERSION, "target": "trapezoidal interval-average recorded electrical power (W)",\n        "random_forest_features": forest["features"], "analytical_features": CORE_FEATURES,\n        "preprocessing": "causal trailing-3 speed median/history; reset at session/outage; fold-fitted imputation",\n        "inference": "predict_power(bundle, canonical_feature_dataframe); raw telemetry requires prepare_session",\n    }, indent=2), encoding="utf-8")\n    summary = {"status": status, "selected_candidate": selected, "warnings": warnings,\n               "development_sessions": split["development_sessions"], "test_sessions": split["test_sessions"],\n               "accepted_rows": len(frame), "development_rows": len(dev), "test_rows": len(test),\n               "accepted_duration_s": float(frame.dt_s.sum()), "test_duration_s": float(test.dt_s.sum()),\n               "model_ranking": sorted(ranking, key=lambda r: r["mean_session_error_pct"]),\n               "environment": {"python": sys.version, "platform": platform.platform(),\n                               "numpy": np.__version__, "pandas": pd.__version__, "scikit_learn": sklearn.__version__,\n                               "joblib": joblib.__version__}}\n    # Explicitly optional and separately named; it has no independent test after this refit.\n    baseline_errors = [r["mean_session_error_pct"] for r in ranking if r["model"] != "random_forest"]\n    rf_error = next(r["mean_session_error_pct"] for r in ranking if r["model"] == "random_forest")\n    improvement = bool(rf_error < min(baseline_errors))\n    summary["forest_beats_all_baselines"] = improvement\n    summary["recommendation"] = "Candidate improves retrospective energy error; obtain a fresh-day test." if improvement else "No improvement over every baseline demonstrated. Keep the best existing baseline."\n    if cfg.refit_all_for_release and improvement:\n        release = train_forest(frame, selected, cfg)\n        release.update({"config": asdict(cfg), "source_manifest": provenance,\n                        "status": "REFIT ON ALL DATA - NEEDS A NEW FUTURE TEST",\n                        "training_scope": "all_sessions_including_previous_test"})\n        joblib.dump(release, out / "random_forest_all_data_UNTESTED.joblib")\n    if cfg.refit_all_for_release and not improvement:\n        summary["warnings"].append("All-data release refit skipped: forest did not beat every analytical baseline.")\n    (out / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False), encoding="utf-8")\n    save_report(out, summary, sessions)\n    try:\n        freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)\n    except (OSError, subprocess.CalledProcessError):\n        freeze = "pip freeze unavailable; see summary.json for package versions\\n"\n    (out / "environment.txt").write_text(freeze, encoding="utf-8")\n    # Copy the actual training implementation so results remain usable away from GitHub.\n    (out / "laval_training.py").write_bytes(Path(__file__).read_bytes())\n    with zipfile.ZipFile(str(out) + ".zip", "w", zipfile.ZIP_DEFLATED) as z:\n        for path in sorted(out.iterdir()):\n            if path.is_file():\n                z.write(path, arcname=path.name)\n    print(json.dumps(summary, indent=2, allow_nan=False))\n    return summary\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("archive")\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--config")\n    parser.add_argument("--reference", help="Optional frozen Front Campus coefficient JSON")\n    args = parser.parse_args()\n    cfg = Config(**json.loads(Path(args.config).read_text())) if args.config else Config()\n    run_workflow(args.archive, args.output, cfg, args.reference)\n\n\nif __name__ == "__main__":\n    main()\n'
BUNDLED_SOURCE_SHA256 = 'ce9dde6c2c644f42ccf34d4daf7d1ecee0a5ae1047d5047753246723f086230e'
FRONT_CAMPUS_REFERENCE_JSON = '{\n  "provenance": {\n    "name": "Front Campus 2026-08-06 transferable vehicle model",\n    "manifest": "data/models/front-campus-2026-08-06.json",\n    "run_count": 3,\n    "lap_count": 15,\n    "training_sample_count": 8115,\n    "current_calibration": {\n      "slope": 0.7949,\n      "intercept_mA": 0.0,\n      "note": "Bench slope applied to historical zero-corrected current. Source CSV values remain unchanged."\n    },\n    "sources": [\n      {\n        "run_id": "front-campus-2026-08-06-run-01",\n        "label": "Front Campus 2026-08-06 Run 1",\n        "gpx": "data/runs/front-campus-2026-08-06-run-01/front-campus-2026-08-06-run-01.gpx",\n        "telemetry": "data/runs/front-campus-2026-08-06-run-01/front-campus-2026-08-06-run-01-telemetry.csv",\n        "split_method": "gate",\n        "expected_laps": 6,\n        "loaded_laps": 6,\n        "raw_telemetry_samples": 4272,\n        "merged_training_samples": 3836,\n        "source_negative_current_samples": 3279,\n        "source_over_20a_samples": 506\n      },\n      {\n        "run_id": "front-campus-2026-08-06-run-02",\n        "label": "Front Campus 2026-08-06 Run 2",\n        "gpx": "data/runs/front-campus-2026-08-06-run-02/front-campus-2026-08-06-run-02.gpx",\n        "telemetry": "data/runs/front-campus-2026-08-06-run-02/front-campus-2026-08-06-run-02-telemetry.csv",\n        "split_method": "gate",\n        "expected_laps": 3,\n        "loaded_laps": 3,\n        "raw_telemetry_samples": 1538,\n        "merged_training_samples": 1375,\n        "source_negative_current_samples": 786,\n        "source_over_20a_samples": 129\n      },\n      {\n        "run_id": "front-campus-2026-08-06-run-03",\n        "label": "Front Campus 2026-08-06 Run 3",\n        "gpx": "data/runs/front-campus-2026-08-06-run-03/front-campus-2026-08-06-run-03.gpx",\n        "telemetry": "data/runs/front-campus-2026-08-06-run-03/front-campus-2026-08-06-run-03-telemetry.csv",\n        "split_method": "gate",\n        "expected_laps": 6,\n        "loaded_laps": 6,\n        "raw_telemetry_samples": 3508,\n        "merged_training_samples": 2904,\n        "source_negative_current_samples": 1550,\n        "source_over_20a_samples": 410\n      }\n    ]\n  },\n  "current_coeffs": [\n    90.34341307680654,\n    472.73651713104965,\n    -10.621379963950767,\n    6047.207614491555,\n    -3454.2345120696073,\n    0.0,\n    0.0,\n    0.0,\n    -11161.70737454799,\n    243.4125495281661,\n    10918.294838436992,\n    -68.72024433024941,\n    172.26089666640658\n  ],\n  "power_coeffs": [\n    2.19435097004979,\n    11.482297264596363,\n    -0.25798269794441164,\n    146.88062574838378,\n    -83.899902063659,\n    0.0,\n    0.0,\n    0.0,\n    -271.10671023022945,\n    5.912247605660822,\n    265.1944635209566,\n    -1.6691460145372288,\n    4.184044919130484\n  ],\n  "ridge": 0.001,\n  "on_current_mA": 19490.948,\n  "cruise_current_mA": 64.3869,\n  "median_voltage_v": 24.289,\n  "coast_sample_count": 439,\n  "transferable": true\n}\n'
assert hashlib.sha256(BUNDLED_TRAINING_SOURCE.encode()).hexdigest() == BUNDLED_SOURCE_SHA256
MODULE_PATH = WORK_DIR / 'laval_training.py'
MODULE_PATH.write_text(BUNDLED_TRAINING_SOURCE, encoding='utf-8')
REFERENCE_PATH = WORK_DIR / 'front_campus_reference.json'
REFERENCE_PATH.write_text(FRONT_CAMPUS_REFERENCE_JSON, encoding='utf-8')
spec = importlib.util.spec_from_file_location('utsm_laval_runtime', MODULE_PATH)
training = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = training
spec.loader.exec_module(training)
print('Training source SHA-256:', BUNDLED_SOURCE_SHA256)
print('Frozen Front Campus source: telemetry commit 00a27b6d25cd143354b17170f522a6a44aaa7b01')


## 3. Configure calibration and data
Leave ARCHIVE_PATH empty to upload the ZIP. Alternatively, enable Drive and enter its mounted file path.
Settings below are examples, not validated sensor measurements.

- **magnitude:** absolute raw current after zero/gain correction (exploratory historical convention).
- **negative:** negative raw current represents consumption. **positive:** positive represents consumption.
- Set current zero/gain using bench records, optionally by filename in CALIBRATION_OVERRIDES.
- CALIBRATION_CONFIRMED only describes your current calibration; constant voltage still limits energy labels.
- Keep September 20 held out. Never tune settings based on its prediction errors.


In [ ]:
ARCHIVE_PATH = os.environ.get("UTSM_ARCHIVE_PATH", "")  # leave blank to upload
USE_GOOGLE_DRIVE = False
SAVE_RESULTS_TO_DRIVE = False
DRIVE_RESULTS_FOLDER = "/content/drive/MyDrive/UTSM-model-results"

CURRENT_MODE = "magnitude"
CURRENT_SCALE = 1.0
CURRENT_ZERO_MA = 0.0
CALIBRATION_CONFIRMED = False
CALIBRATION_OVERRIDES = {}  # {"2026-09-19-19-26.csv": {"current_mode": "negative", "current_scale": 1.0}}
TEST_DATES = ["2026-09-20"]
REFIT_ALL_FOR_RELEASE = False  # requires calibration + improvement; still needs a future test

cfg = training.Config(
    current_mode=CURRENT_MODE, current_scale=CURRENT_SCALE,
    current_zero_mA=CURRENT_ZERO_MA, calibration_confirmed=CALIBRATION_CONFIRMED,
    calibration_overrides=CALIBRATION_OVERRIDES, test_dates=TEST_DATES,
    refit_all_for_release=REFIT_ALL_FOR_RELEASE,
)
cfg.validate()
if not CALIBRATION_CONFIRMED:
    print("EXPLORATORY RUN: current calibration is not confirmed.")


## 4. Provide your ZIP (source files remain unchanged)


In [ ]:
if USE_GOOGLE_DRIVE or SAVE_RESULTS_TO_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Drive mounting is a Colab option; use a local path outside Colab.")
    from google.colab import drive
    drive.mount("/content/drive")

if not ARCHIVE_PATH:
    if not IN_COLAB:
        raise ValueError("Set ARCHIVE_PATH to your ZIP outside Colab.")
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("Upload exactly one telemetry ZIP.")
    archive_path = WORK_DIR / Path(zip_names[0]).name
    archive_path.write_bytes(uploaded[zip_names[0]])
else:
    archive_path = Path(ARCHIVE_PATH).expanduser().resolve()
if not archive_path.is_file():
    raise FileNotFoundError(archive_path)
print("Archive:", archive_path.name)


## 5. Inspect retained data and the session split
This uses embedded CSV GPS, not the untimed track GPX. History resets at gaps/reboots;
invalid sensor intervals are excluded. Quality rules are fixed before model tuning.

These outputs are **accepted moving intervals**, not complete-lap labels or total session energy.
Review excluded files, accepted duration, current signs, voltage and sensor disagreement.
Do not loosen rules just because one model's score improves.


In [ ]:
import pandas as pd
try:
    from IPython.display import display, Image
except ImportError:
    display = print
canonical, quality, provenance = training.load_archive(archive_path, cfg)
dev, held_out, split = training.split_sessions(canonical, cfg)
display(quality)
display(pd.DataFrame([
    {"split": "development", "sessions": dev.session_id.nunique(), "rows": len(dev), "valid_minutes": dev.dt_s.sum()/60},
    {"split": "held_out", "sessions": held_out.session_id.nunique(), "rows": len(held_out), "valid_minutes": held_out.dt_s.sum()/60},
]))
print("Archive SHA-256:", provenance["archive_sha256"])
print(json.dumps(split, indent=2))
if quality.get("constant_voltage", pd.Series(dtype=bool)).fillna(False).any():
    print("CONSTANT VOLTAGE: energy labels assume the recorded voltage is correct; voltage sag is unobserved.")


## 6. Train, tune and compare
Four bounded forest candidates use session-grouped cross-validation on development data only.
Imputation and analytical scaling are fit inside each training fold.
After selecting the forest, evaluate the held-out sessions once against:
1. the selected Random Forest,
2. an analytical regression refit on the same development data,
3. the frozen Front Campus analytical model.

A new output directory is used; previous results are never overwritten. Reusing this date's
scores to choose further changes invalidates it as a final test.


In [ ]:
from datetime import datetime, timezone
run_name = "laval-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
output_dir = Path(os.environ.get("UTSM_OUTPUT_DIR", str(WORK_DIR / "outputs" / run_name))).resolve()
summary = training.run_workflow(archive_path, output_dir, cfg, REFERENCE_PATH)
print("Decision:", summary["recommendation"])
print("Results:", output_dir)


## 7. Read the results
Lower error is better. Session errors cover only accepted driving intervals.
Time-bin MAE is diagnostic; it is not lap-energy accuracy.

The archive has already been explored in an audit. A real fresh-day test and verified calibration
are still required before claiming race performance or deployment readiness.


In [ ]:
display(pd.DataFrame(summary["model_ranking"]))
display(pd.read_csv(output_dir / "held_out_sessions.csv"))
if "Image" in globals():
    display(Image(filename=str(output_dir / "comparison.png")))
for warning in summary["warnings"]:
    print("-", warning)


## 8. Reload the forest and test inference
This proves the saved artifact includes the fitted imputer and ordered schema.
The prediction call below uses only input features, not current/power/energy labels.
New raw data must use the same causal preprocessing and calibration.


In [ ]:
import joblib
import numpy as np
model = joblib.load(output_dir / "random_forest_development.joblib")
examples = held_out[model["features"]].head(5)
predicted_watts = training.predict_power(model, examples)
display(examples.assign(predicted_power_w=predicted_watts))
saved_predictions = pd.read_csv(output_dir / "held_out_predictions.csv.gz")
recorded = saved_predictions[saved_predictions.model == "random_forest"].predicted_power_w.head(5).to_numpy()
np.testing.assert_allclose(predicted_watts, recorded)
print("Saved-model prediction round trip passed.")


## 9. Save/download everything
The ZIP includes models, canonical data, predictions, settings, hashes, source code, reports and the
actual environment. Save it before the runtime ends. No raw telemetry is sent to GitHub.

The notebook does not replace an existing deployed model. An all-data refit is separately named,
off by default, and skipped unless calibration is confirmed and the forest beats all analytical baselines.


In [ ]:
import shutil
result_zip = Path(str(output_dir) + ".zip")
if SAVE_RESULTS_TO_DRIVE:
    destination = Path(DRIVE_RESULTS_FOLDER) / output_dir.name
    destination.mkdir(parents=True, exist_ok=False)
    shutil.copy2(result_zip, destination / result_zip.name)
    print("Saved to Drive:", destination)
if IN_COLAB:
    from google.colab import files
    files.download(str(result_zip))
else:
    print("Download/share this result bundle:", result_zip)


## Next experiments
First verify sensor gain/polarity and fixed voltage. Then collect lap boundaries and throttle commands.
Keep future-day data independent. Only after a clean CPU baseline should GPU/sequence-network
experiments consume Colab credits.

The notebook source is maintained by tools/build_laval_notebook.py; tests check that its embedded
helper, analytical reference and requirements match the reviewed source files.
